# Exploratory Data Analysis — Telco Customer Churn\n\nThis notebook explores the IBM Telco Customer Churn dataset to understand churn drivers before model training.\n\n**Dataset path:** data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.preprocessing import get_default_data_path, load_raw_data

sns.set_theme(style='whitegrid', context='notebook')
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = get_default_data_path()
print(f'Dataset path: {DATA_PATH}')
print(f'Exists: {DATA_PATH.exists()}')


## 1. Dataset Overview

In [ ]:
df = load_raw_data(DATA_PATH)
print('Shape:', df.shape)
print()
df.info()
display(df.head())


In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print()
print('Blank TotalCharges rows:', (df['TotalCharges'].astype(str).str.strip() == '').sum())
print('Duplicate rows:', df.duplicated().sum())
print()
print('Target distribution:')
print(df['Churn'].value_counts())
print()
print('Target rate:')
print(df['Churn'].value_counts(normalize=True).round(3))


### Overview Insights

- The dataset has ~7,000 customers and ~21 columns.
- TotalCharges may contain blank strings for brand-new customers (tenure = 0); these should be coerced to numeric and imputed during preprocessing.
- customerID is an identifier and must be excluded from modeling features.
- Churn is imbalanced (roughly 25–30% churners), so stratified splitting and class-aware metrics (ROC-AUC, recall) are important.


## 2. Customer Analysis

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

def churn_rate(series):
    return (series == 'Yes').mean()

print('Churn rate by Contract:')
print(df.groupby('Contract')['Churn'].apply(churn_rate).sort_values(ascending=False).round(3))
print()
print('Churn rate by InternetService:')
print(df.groupby('InternetService')['Churn'].apply(churn_rate).sort_values(ascending=False).round(3))
print()
print('Churn rate by PaymentMethod:')
print(df.groupby('PaymentMethod')['Churn'].apply(churn_rate).sort_values(ascending=False).round(3))
print()
print('Churn rate by TechSupport:')
print(df.groupby('TechSupport')['Churn'].apply(churn_rate).sort_values(ascending=False).round(3))
print()
print('Mean tenure by Churn:')
print(df.groupby('Churn')['tenure'].mean().round(2))
print()
print('Mean MonthlyCharges by Churn:')
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))


### Customer Analysis Insights

- **Contract:** Month-to-month customers churn far more than one-year or two-year contracts.
- **Tenure:** Customers who leave tend to have shorter tenure.
- **Monthly charges:** Churners typically pay higher monthly charges on average.
- **Internet service:** Fiber optic users show elevated churn relative to DSL / no internet.
- **Payment method:** Electronic check is associated with higher churn.
- **Tech support:** Customers without tech support churn more often than those with support.


## 3. Visualizations

In [ ]:
# 1. Churn Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=df, x='Churn', order=df['Churn'].value_counts().index, ax=ax, palette='Set2')
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'churn_distribution.png', dpi=150)
plt.show()


In [ ]:
# 2. Churn by Contract Type
fig, ax = plt.subplots(figsize=(8, 5))
rates = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
rates.plot(kind='bar', ax=ax, color='#e76f51')
ax.set_title('Churn Rate by Contract Type')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Contract')
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'churn_by_contract.png', dpi=150)
plt.show()


In [ ]:
# 3. Churn by Internet Service
fig, ax = plt.subplots(figsize=(8, 5))
rates = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
rates.plot(kind='bar', ax=ax, color='#457b9d')
ax.set_title('Churn Rate by Internet Service')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Internet Service')
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'churn_by_internet.png', dpi=150)
plt.show()


In [ ]:
# 4. Monthly Charges Distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, ax=ax, palette='muted')
ax.set_title('Monthly Charges Distribution by Churn')
ax.set_xlabel('Monthly Charges')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'monthly_charges_distribution.png', dpi=150)
plt.show()


In [ ]:
# 5. Tenure vs Churn
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='tenure', ax=ax, palette='pastel')
ax.set_title('Tenure vs Churn')
ax.set_xlabel('Churn')
ax.set_ylabel('Tenure (months)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'tenure_vs_churn.png', dpi=150)
plt.show()


In [ ]:
# 6. Correlation Heatmap
numeric = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
numeric['Churn'] = (df['Churn'] == 'Yes').astype(int)
corr = numeric.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation Heatmap (Numerical Features + Churn)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()


## 4. Modeling Implications

1. Prefer **stratified** train/test splits due to class imbalance.
2. Encode categoricals with one-hot encoding; scale numerical features.
3. Treat blank TotalCharges as missing and impute using training-fold statistics only.
4. Strong churn signals: short tenure, month-to-month contracts, fiber optic, electronic check, no tech support.
5. Primary model selection metric: **ROC-AUC** (threshold-independent ranking quality).
